[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week6/encoder_applications_demo.ipynb)

# Encoder applications — hands-on exploration

**PSYC 51.17: Models of language and communication**  
**Week 6**

---

## Learning objectives

By the end of this session, you will:
1. Apply BERT-based models to text classification and sentiment analysis
2. Extract and visualize named entities using token classification pipelines
3. Implement extractive question answering to find information in passages
4. Systematically measure and visualize gender bias across different professions

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q transformers torch matplotlib numpy datasets

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
from transformers import pipeline

warnings.filterwarnings('ignore')

print("\u2713 All imports successful!")

## Part 1: Text classification with BERT

Text classification is one of the most common applications of encoder models. By adding a classification head on top of BERT's `[CLS]` token, we can fine-tune the model for tasks like sentiment analysis. We'll use a DistilBERT model fine-tuned on the SST-2 (Stanford Sentiment Treebank) dataset.

In [ ]:
# Load the sentiment analysis pipeline
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "This course on LLMs is absolutely fascinating!",
    "I found the lecture a bit confusing, but the demo helped.",
    "The weather today is quite gloomy.",
    "I had a great time at the park, though it was crowded.",
    "The movie was okay, but the ending felt rushed."
]

print("Sentiment Analysis Results:")
print("=" * 60)
for sent in sentences:
    result = classifier(sent)[0]
    print(f"Sentence: {sent}")
    print(f"  Label: {result['label']} (Score: {result['score']:.3f})")
    print()

# Adversarial examples: appending unrelated negative words
print("Adversarial Examples (appending negative words):")
print("-" * 60)
adversarial_sent = "I love this movie. terrible bad awful"
result = classifier(adversarial_sent)[0]
print(f"Sentence: {adversarial_sent}")
print(f"  Label: {result['label']} (Score: {result['score']:.3f})")

In [ ]:
# YOUR TURN! Try your own sentences.
# Can you find a sentence that tricks the model?

my_sentences = [
    "The food was delicious but the service was slow.",
    # Add more sentences here!
]

for sent in my_sentences:
    result = classifier(sent)[0]
    print(f"Sentence: {sent}")
    print(f"  Label: {result['label']} (Score: {result['score']:.3f})")
    print()

### 💡 Discussion

- How does the model handle sentences with both positive and negative elements (e.g., "delicious but slow")?
- Why did the adversarial example change the prediction? What does this suggest about how the model "understands" sentiment?
- Are there specific words that seem to carry more weight in the model's decision?
- How might you use this for a real-world application, like monitoring social media trends?

## Part 2: Named entity recognition (NER)

Named Entity Recognition is a **token classification** task where the model assigns a label to each token in a sentence. Common labels include Person (PER), Organization (ORG), and Location (LOC). We use the **BIO tagging** scheme (Beginning, Inside, Outside) to handle multi-token entities:
- **B-PER**: Beginning of a person's name.
- **I-PER**: Inside of a person's name.
- **O**: Outside of any entity.

In [ ]:
# Load the NER pipeline
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

sentences = [
    "Albert Einstein was born in Ulm, Germany, and later moved to the United States.",
    "Google was founded by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University.",
    "The United Nations is headquartered in New York City."
]

def visualize_ner(text, entities):
    """Simple visualization of entities by highlighting them in the text."""
    sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)
    highlighted_text = text
    for ent in sorted_entities:
        start, end, label = ent['start'], ent['end'], ent['entity_group']
        highlighted_text = highlighted_text[:start] + f"[{text[start:end]}]({label})" + highlighted_text[end:]
    return highlighted_text

print("NER Results:")
print("=" * 60)
for sent in sentences:
    entities = ner(sent)
    print(visualize_ner(sent, entities))
    print()

### 💡 Discussion

- Why do we need 'B-' and 'I-' prefixes? What would happen if we just used 'PER'?
- Did the model correctly identify multi-word entities like "United States" or "Larry Page"?
- Try a sentence with an ambiguous name (e.g., "Apple" as a company vs a fruit). How does the model perform?
- How could NER be used to automatically summarize news articles or organize documents?

## Part 3: Question answering

Extractive Question Answering involves finding the exact span of text in a passage that answers a given question. BERT models are often fine-tuned on the SQuAD (Stanford Question Answering Dataset) for this task.

In [ ]:
# Load the QA pipeline
qa_model = pipeline("question-answering", model="bert-large-uncased-whole-word-masking-finetuned-squad")

context = """
The Dartmouth course 'Models of Language and Communication' (PSYC 51.17) explores the intersection of psychology, 
linguistics, and computer science. Students learn about the evolution of chatbots, from ELIZA in 1966 to modern 
Large Language Models like GPT-4. The course is taught by Professor Jeremy Manning and covers topics such as 
tokenization, word embeddings, and transformer architectures. Assignments include building a SPAM classifier 
and creating a customer service chatbot.
"""

questions = [
    "What is the course number for Models of Language and Communication?",
    "Who teaches the course?",
    "When was ELIZA created?",
    "What are some topics covered in the course?",
    "What is the capital of France?"  # Unanswerable from context
]

print("Question Answering Results:")
print("=" * 60)
for q in questions:
    result = qa_model(question=q, context=context)
    print(f"Question: {q}")
    print(f"  Answer:   {result['answer']}")
    print(f"  Span:     ({result['start']}, {result['end']})")
    print(f"  Confidence: {result['score']:.3f}")
    print()

### 💡 Discussion

- How does the model handle the "unanswerable" question? Look at the confidence score.
- Does the model return a complete sentence or just the specific answer span?
- Try rephrasing a question (e.g., "Who is the instructor?"). Does the answer change?
- What are the limitations of extractive QA compared to generative QA (like ChatGPT)?

## Part 4: Measuring bias systematically

Encoder models like BERT can reveal societal biases present in their training data. We can use the **Masked Language Modeling (MLM)** task to probe these biases by comparing the probabilities of different tokens in specific contexts. We'll use the **SAGED** (Stereotype and Gender Encoding Discovery) methodology to look at gender bias across professions.

In [ ]:
# Load the fill-mask pipeline
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

professions = [
    "doctor", "nurse", "engineer", "teacher", "scientist", 
    "lawyer", "secretary", "pilot", "flight attendant", "programmer", 
    "librarian", "chef", "janitor", "manager", "assistant"
]

results = []
for prof in professions:
    template = f"The {prof} said [MASK] would be right back."
    preds = fill_mask(template)
    
    he_score = next((p['score'] for p in preds if p['token_str'] == 'he'), 0)
    she_score = next((p['score'] for p in preds if p['token_str'] == 'she'), 0)
    
    # Calculate ratio (log ratio is often better for visualization)
    # We add a small epsilon to avoid division by zero
    ratio = he_score / (she_score + 1e-9)
    results.append({"profession": prof, "he": he_score, "she": she_score, "ratio": ratio})

# Sort by ratio
results.sort(key=lambda x: x['ratio'])

profs = [r['profession'] for r in results]
ratios = [r['ratio'] for r in results]

# Plotting
plt.figure(figsize=(12, 8))
colors = ['#9d162e' if r > 1 else '#267aba' for r in ratios]
plt.barh(profs, ratios, color=colors)
plt.axvline(x=1, color='#6f42c1', linestyle='--', alpha=0.5, label='Neutral')
plt.xlabel("He/She Probability Ratio (values > 1 favor 'he')", fontsize=12)
plt.title("Gender Bias in BERT Predictions across Professions", fontsize=14)
plt.grid(axis='x', color='#00693e', alpha=0.1)
plt.legend()

# Calculate stats
mean_ratio = np.mean(ratios)
var_ratio = np.var(ratios)
print(f"Mean Bias Ratio: {mean_ratio:.3f}")
print(f"Bias Variance:   {var_ratio:.3f}")
plt.show()

### 💡 Discussion

- Which professions show the strongest bias towards 'he'? Which towards 'she'?
- Does this align with real-world labor statistics or societal stereotypes?
- How does the **SAGED** methodology help us quantify these biases systematically?
- What are the ethical implications of using biased models in applications like automated hiring or content moderation?

## Summary

| Part | Application | Key Mechanism |
|------|-------------|---------------|
| Text Classification | Sentiment Analysis | `[CLS]` token + classification head |
| Token Classification | NER | Per-token labeling (BIO tags) |
| Question Answering | Extractive QA | Finding answer spans in context |
| Bias Measurement | MLM Probing | Comparing token probabilities in templates |

## Further exploration

1. **Zero-shot classification**: Try the `zero-shot-classification` pipeline. How does it classify text into labels it hasn't seen during training?
2. **Multilingual NER**: Use a multilingual model (e.g., `dbmdz/bert-base-turkish-cased`) to perform NER on non-English text.
3. **SQuAD 2.0**: Compare models trained on SQuAD 1.1 vs SQuAD 2.0. How do they differ in handling unanswerable questions?
4. **De-biasing**: Research techniques like "Counterfactual Data Augmentation" (CDA). How can we reduce the biases we measured in Part 4?